In [20]:

import os
import pandas as pd
from collections import defaultdict

guide_file = "01_make_simulations_output/guides.txt"
sample_file = "01_make_simulations_output/samples.txt"
aggregated_stats_file = "02_run_output/aggregated_stats_all.txt"

samples = pd.read_csv(sample_file, sep="\t")
samples

,Name,r1
0,sample_0_edited_0_noise_rep0,01_make_simulations_output/sample_0_edited_0_n...
1,sample_1_unedited_0_noise_rep0,01_make_simulations_output/sample_1_unedited_0...
2,sample_2_edited_0.01_noise_rep0,01_make_simulations_output/sample_2_edited_0.0...
3,sample_3_unedited_0.01_noise_rep0,01_make_simulations_output/sample_3_unedited_0...
4,sample_4_edited_0.05_noise_rep0,01_make_simulations_output/sample_4_edited_0.0...
5,sample_5_unedited_0.05_noise_rep0,01_make_simulations_output/sample_5_unedited_0...
6,sample_6_edited_0.1_noise_rep0,01_make_simulations_output/sample_6_edited_0.1...
7,sample_7_unedited_0.1_noise_rep0,01_make_simulations_output/sample_7_unedited_0...
8,sample_8_edited_0_noise_rep1,01_make_simulations_output/sample_8_edited_0_n...
9,sample_9_unedited_0_noise_rep1,01_make_simulations_output/sample_9_unedited_0...


In [ ]:
background_mutation_rates = [0, 0.01, 0.05, 0.1]

output_folder = "03_make_replots.ipynb.output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

sig_tests = [("hard_cutoff","hard_cutoff,0.01"),
             ("mean_diff","mean_diff,edited,unedited,0.01"),
             ("t_test","t_test,edited,unedited,0.05"),
             ("mann_whitney","mann_whitney,edited,unedited,0.05"),
             ("neg_binomial","neg_binomial,edited,unedited,0.05")]

command_file = os.path.join(output_folder, "commands.txt")
sig_test_outputs = defaultdict(list)
with open(command_file, 'w') as f:
    for background_mutation_rate in background_mutation_rates:
        edited_samples = [x for x in samples['Name'] if f"_edited_{background_mutation_rate}_noise" in x]
        unedited_samples = [x for x in samples['Name'] if f"_unedited_{background_mutation_rate}_noise" in x]
        background_file = os.path.join(output_folder, f"noise_{background_mutation_rate}.agg_samples.txt")
        with open(background_file, 'w') as bg_f:
            bg_f.write("Name\tfastq_r1\tGroup\n")
            for sample in edited_samples:
                bg_f.write(f'{sample}\t{samples.loc[samples["Name"] == sample, "r1"].values[0]}\tedited\n')
            for sample in unedited_samples:
                bg_f.write(f'{sample}\t{samples.loc[samples["Name"] == sample, "r1"].values[0]}\tunedited\n')
        f.write(f'CRISPRessoSea Replot --reordered_stats_file {aggregated_stats_file} --reordered_sample_file {background_file} --output_folder {output_folder}/noise_{background_mutation_rate}\n')
        for sig_test_name, sig_test_string in sig_tests:
            test_output_folder = os.path.join(output_folder, f"noise_{background_mutation_rate}_{sig_test_name}")
            f.write(f'CRISPRessoSea Replot --reordered_stats_file {aggregated_stats_file} --reordered_sample_file {background_file} --output_folder {test_output_folder} --sig_method_parameters {sig_test_string}\n')
            sig_test_outputs[sig_test_name].append(test_output_folder)

In [ ]:
aggregated_stats = pd.read_csv(aggregated_stats_file, sep="\t").copy()
aggregated_stats["editing_rate"] = aggregated_stats["target_name"].str.split("EditingRate").str[-1].astype(float)

sig_test_summary_tables = {}

for sig_test_name, test_output_folders in sig_test_outputs.items():
    summary_rows = []

    for test_output_folder in test_output_folders:
        background_noise_rate = float(test_output_folder.split("noise_")[-1].split("_")[0])

        if sig_test_name == "hard_cutoff":
            edited_samples = [
                sample_name
                for sample_name in samples["Name"]
                if f"_edited_{background_noise_rate}_noise" in sample_name
            ]
            unedited_samples = [
                sample_name
                for sample_name in samples["Name"]
                if f"_unedited_{background_noise_rate}_noise" in sample_name
            ]

            edited_cols = [f"{sample_name}_mod_pct" for sample_name in edited_samples]
            unedited_cols = [f"{sample_name}_mod_pct" for sample_name in unedited_samples]

            hard_cutoff_df = aggregated_stats[["target_name", "target_id", "editing_rate"] + edited_cols + unedited_cols].copy()
            hard_cutoff_df["background_noise_rate"] = background_noise_rate
            hard_cutoff_df["edited_significant_n"] = (hard_cutoff_df[edited_cols] >= 0.01).sum(axis=1)
            hard_cutoff_df["unedited_significant_n"] = (hard_cutoff_df[unedited_cols] >= 0.01).sum(axis=1)
            hard_cutoff_df["pvalue"] = pd.NA
            hard_cutoff_df["significant"] = pd.NA

            summary_rows.append(
                hard_cutoff_df[[
                    "target_name",
                    "target_id",
                    "editing_rate",
                    "background_noise_rate",
                    "edited_significant_n",
                    "unedited_significant_n",
                    "pvalue",
                    "significant",
                ]]
            )
            continue

        p_value_file = os.path.join(test_output_folder, "CRISPRessoSea_all_mod_pct.p_values.txt")
        if not os.path.exists(p_value_file):
            print(f"Skipping missing file: {p_value_file}")
            continue

        p_value_df = pd.read_csv(p_value_file, sep="\t")
        p_value_df["editing_rate"] = p_value_df["target_name"].str.split("EditingRate").str[-1].astype(float)
        p_value_df["background_noise_rate"] = background_noise_rate
        p_value_df["pvalue"] = p_value_df["p_val"] if "p_val" in p_value_df.columns else pd.NA
        p_value_df["edited_significant_n"] = pd.NA
        p_value_df["unedited_significant_n"] = pd.NA

        summary_rows.append(
            p_value_df[[
                "target_name",
                "target_id",
                "editing_rate",
                "background_noise_rate",
                "edited_significant_n",
                "unedited_significant_n",
                "pvalue",
                "significant",
            ]]
        )

    sig_test_summary_tables[sig_test_name] = pd.concat(summary_rows, ignore_index=True).sort_values(
        ["background_noise_rate", "editing_rate"]
    )
    sig_test_summary_tables[sig_test_name].to_csv(
        os.path.join(output_folder, f"{sig_test_name}.aggregated_significance.tsv"),
        sep="\t",
        index=False,
    )

sig_test_summary_tables["hard_cutoff"].head()


In [ ]:
{sig_test_name: df.head() for sig_test_name, df in sig_test_summary_tables.items()}
